In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
NB_BOOT = 4000
STT_SEG = (130, 215)        # Q7-F/I/I″ 승계

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
KS         = (8, 16)        # Q7-I 승계
MIN_S_TPL  = 20             # Q7-F/H/I 승계
K_FOLD     = 5              # Q7-H 승계 (R22)
N_REPEAT   = 3              # Q7-H 승계
BANDS      = (0.05, 0.03, 0.02, 0.01)   # Q7-I″ 승계 — **고정 코호트로 다시 그린다**
BAND_MIN   = 4              # ★ Q7-I″ 가 쓴 값. 이번엔 **묶였는지 실측**한다
MIN_BAND_S, MIN_BAND_N = 3, 3
MIN_MATCH_S, MIN_MATCH_PAIR = 20, 200
MIN_MATCH_REC = 15
N_SHUF     = 5              # ★ 정합 조건 라벨셔플 null 반복
EXACT_TIE  = 0.505          # K1 — 정확 정합이면 f1 은 동점뿐이라 0.5 여야 한다
BONF2      = 0.0125         # 양측 0.05/2대조 → 각 쪽 꼬리
ISO_HI, ISO_LO = 0.7, 0.3   # Q7-F/I 승계

CONFIG = dict(
    exp="quest46_q7i3_match_exact", quest="ailab-2026-0046", step="svdb-match-exact",
    parent_exp=["quest46_q7i2_match_residual", "ailab-2026-0059"],
    purpose=("Q7-I″ 의 J1 은 바닥을 **개체별 max(f1, STT)** 로 잡고 문턱을 0 으로 뒀다 — "
             "잡음 둘의 최댓값은 위로 뜬다(0.6440 vs 성분 0.5331·0.5708). 기각 −0.0705 는 "
             "전부 그 팽창이었다(=−0.0733). 이번엔 **max 를 안 쓰고**, 각 팔을 **자기 "
             "라벨셔플 영분포 위의 초과분**으로 환산해 비교한다. 동시에 **정합기 자체를 "
             "감사**한다 — `pre_rr` 격자·BAND_MIN 이 묶었는지·짝 단위 잔여·고정 코호트"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    bands=list(BANDS), ks=list(KS), n_shuffle=N_SHUF,
    predictions={
        "K0": "(관문 아님) pre_rr 격자 Δ · 실효 대역폭 bw/Δ — BAND_MIN 이 묶었나",
        "K1": f"★ **정확 정합**(pre_rr 값 자체로 짝) 에서 f1ₘ CI 상한 < {EXACT_TIE} "
              "— 동점뿐이라 0.5 여야 한다. 아니면 **정합기 결함 확정**",
        "K2": "(관문 아님) 정확 정합 조건 전 팔 매크로 — ★ f2 를 반드시 낸다",
        "K3": "정확 정합에서 짝지은 f2_16ₘ − f1ₘ CI 하한 > 0  "
              "[**절대 RR 을 맞춰도 상대 조기성은 안 맞는다**]",
        "K4": "정확 정합에서 (f3ₘ − 0.5) − (STTₘ − nullSTTₘ) CI 하한 > 0 (Bonferroni 2) "
              "[J1 재판정 · max 바닥 폐기]",
        "K5": "정확 정합에서 LR(전부)ₘ − nullLR(전부)ₘ CI 하한 > 0 "
              "[대역에 안 움직이던 0.778 이 교차적합 바닥인가]",
        "K6": "혼합 층·런 층 **따로** 짝지은 f2_16ₘ − f1ₘ  [J4 분리 재판정]",
        "K7": "짝지은 LR(f3,f4,f5)ₘ − LR(f3,f4)ₘ CI 하한 > 0  "
              "[f5 의 짝은 f1 이 아니라 f3 다 — 3박자 패턴의 나머지 절반]"},
    caveat=("**Q7-I″ 의 J1 「❌ 기각」은 인공물이므로 인용하지 않는다**(→ `ailab-2026-0059` §2). "
            "정확 정합이 성립 안 하면(격자가 없거나 빈이 너무 작으면) K1 은 「측정 불가」로 "
            "내고 **그 아래 관문을 읽지 않는다** — 대역 정합으로 몰래 되돌리지 않는다(R16). "
            "개체 내부 로지스틱은 **라벨을 쓰는 상한**이다. 학습 0회."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7i3_match_exact", CONFIG, project=PROJECT)
run.log("설정 ✅ 정확 정합 + 자기 영분포 초과분 · 셔플 " + str(N_SHUF))

In [ ]:
# CELL 2 — 【K-0a】 자산 · 매핑 (Q7-D/E/F/H/I/I″ 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BSTT = np.ascontiguousarray(np.asarray(d5["beat"])[keep][:, :, STT_SEG[0]:STT_SEG[1]]).astype("float32")
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【K-0a】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【K-0b】 ★ 정합기 감사 ① — `pre_rr` 격자와 실효 대역폭
# Q7-I″ 는 `bw = max(BAND_MIN, bf·중앙RR)` 을 썼다. BAND_MIN 이 묶으면
# 라벨은 0.010 인데 실제로는 더 넓다 — 곡선의 마지막 점이 거짓말이 된다.
run.log("\n" + "=" * 100)
run.log("【K-0b】 정합기 감사 ① — pre_rr 격자 · BAND_MIN 이 묶었나")
run.log("=" * 100)
GRID = {}
for r in ALLR:
    pv = PRE[REC == r]
    uq = np.unique(np.round(pv, 6))
    GRID[int(r)] = dict(med=float(np.median(pv)), n_uniq=int(len(uq)),
                        grid=float(np.median(np.diff(uq))) if len(uq) > 1 else float("nan"))
GMED = float(np.nanmedian([g["grid"] for g in GRID.values()]))
MEDS = np.array([g["med"] for g in GRID.values()])
run.log(f"  pre_rr 고유값 간격(격자) 중앙 **Δ = {GMED:.4f} 샘플** · "
        f"개체당 고유값 중앙 {np.median([g['n_uniq'] for g in GRID.values()]):.0f}")
run.log(f"  중앙 RR 중앙 {np.median(MEDS):.1f} 샘플 (360Hz 기준 {np.median(MEDS)/360:.3f}s)")
run.log("\n  대역 라벨 vs **실효** 대역폭 (bw = max(BAND_MIN, bf·중앙RR))")
BINDS = {}
for bf in BANDS:
    bw = np.maximum(BAND_MIN, bf * MEDS)
    eff = bw / np.maximum(MEDS, 1e-9)
    nb_ = int((bw <= BAND_MIN + 1e-9).sum())
    BINDS[f"{bf:.3f}"] = dict(eff_med=float(np.median(eff)), n_bound=nb_,
                              bw_over_grid=float(np.median(bw)) / max(GMED, 1e-9))
    flag = "  ⛔ **BAND_MIN 이 묶었다**" if nb_ > len(MEDS) / 2 else ""
    run.log(f"    라벨 {bf:.3f} → 실효 **{np.median(eff):.4f}** · "
            f"BAND_MIN 에 묶인 개체 {nb_}/{len(MEDS)} · "
            f"빈폭/격자 **{np.median(bw)/max(GMED,1e-9):.2f}배**{flag}")
run.log("\n  ▸ 빈폭/격자 > 1 이면 한 빈에 **서로 다른 RR 값이 섞인다** → 그 안에서 S 가")
run.log("    짧은 쪽에 몰릴 수 있고, 그게 `f1` 이 0.5 로 안 가는 이유일 수 있다.")
run.log("    → **K1 은 격자 자체를 빈으로 쓴다**(정확 정합). 그러면 동점뿐이라 0.5 여야 한다")
CONFIG["grid"] = dict(delta=GMED, med_rr=float(np.median(MEDS)), bands=BINDS)
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【K-A】 특징·점수 (Q7-I/I″ 승계 · 여기서 재설계하지 않는다)
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def rhythm_feats(pre_v, post_v, ks):
    """Q7-I 승계 + **f6 신설**(국소 기저선의 전역 대비 이동 = 리듬 전환 지표 · 라벨프리)."""
    med = float(np.median(pre_v)); F, NM = [], []
    F.append(med - pre_v); NM.append("f1")
    b_first = None
    for k in ks:
        b = local_base(pre_v, k)
        if b_first is None:
            b_first = b
        F.append(1.0 - pre_v / np.maximum(b, 1e-9)); NM.append(f"f2_{k}")
    F.append(1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)); NM.append("f3")
    cv = np.empty(len(pre_v))
    for i in range(len(pre_v)):
        a = max(0, i - ks[0]); w = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w) / max(np.mean(w), 1e-9))
    F.append(cv); NM.append("f4")
    F.append(np.r_[0.0, F[1][:-1]]); NM.append("f5")
    b_last = local_base(pre_v, ks[-1])
    F.append(1.0 - b_last / max(med, 1e-9)); NM.append("f6")     # ★ 런 안이면 크다
    return np.stack(F, axis=1), NM

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

def build_scores(X, NAMES, tt, B_stt, seed):
    """모든 팔의 비트별 점수. 라벨셔플 null 도 **같은 함수**로 만든다(공정)."""
    S = {nm_: X[:, j] for j, nm_ in enumerate(NAMES)}
    i3, i4, i5 = NAMES.index("f3"), NAMES.index("f4"), NAMES.index("f5")
    XI = np.c_[X[:, [i3, i4, i5]],
               (X[:, i3] - X[:, i3].mean()) * (X[:, i5] - X[:, i5].mean())]   # ★ f3×f5
    arms = {"lr_all": X,
            "lr_norr": X[:, [i3, i4]],
            "lr_norr5": X[:, [i3, i4, i5]],
            "lr_i35": XI,
            "lr_f1": X[:, [NAMES.index("f1")]],
            "lr_f1f6": X[:, [NAMES.index("f1"), NAMES.index("f6")]]}
    for nm_, XX in arms.items():
        sc_ = cv_logit(XX, tt, K_FOLD, seed, N_REPEAT)
        if sc_ is None:
            return None
        S[nm_] = sc_
    st = two_template_cv(B_stt, tt, K_FOLD, seed, N_REPEAT)
    S["stt"] = st if st is not None else np.full(len(tt), np.nan)
    return S

run.log("\n" + "=" * 100)
run.log("【K-A】 특징·점수 계산 (+ f6 국소기저선 이동 · f3×f5)")
run.log("=" * 100)
SCORES, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())} — 최소 {MIN_S_TPL} 미달")); continue
    pre_m, post_m = PRE[mm], POST[mm]
    X, NAMES = rhythm_feats(pre_m, post_m, KS)
    S = build_scores(X, NAMES, tt, BSTT[mm], SEED0)
    if S is None:
        SKIP.append((int(r), "겹 안 클래스 부족")); continue
    iso_f, mx_run = run_struct(tt)
    SCORES[int(r)] = S
    META[int(r)] = dict(tt=tt, pre=pre_m, X=X, B=BSTT[mm], n=int(len(mm)),
                        pos=int(tt.sum()), prev=float(tt.mean()),
                        iso_frac=iso_f, max_run=mx_run)
RS = sorted(SCORES)
run.log(f"  채점 {len(RS)}개체 · 제외 {len(SKIP)}개체")
if len(RS) < 10:
    raise AssetError("채점된 개체가 너무 적다")
ARMS = ["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6",
        "lr_f1", "lr_f1f6", "lr_all", "lr_norr", "lr_norr5", "lr_i35", "stt"]
RAW = {a: np.array([roc_auc_score(META[r]["tt"].astype(int), SCORES[r][a])
                    if np.isfinite(SCORES[r][a]).all() else np.nan for r in RS]) for a in ARMS}
run.log("  무정합 매크로 — " + " · ".join(f"{a} {np.nanmean(RAW[a]):.4f}" for a in
        ("f1", "f2_16", "f3", "f6", "lr_f1", "lr_all", "stt")))
NAMES_G = NAMES
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【K-B】 ★ 정확 정합 — 짝은 `pre_rr` **값 자체**로 짓는다
def grouped_auc(sc, tt, key, min_s, min_n, pre_v=None):
    """`key` 가 같은 비트끼리만 쌍을 센다. **평가만 제한**(Q7-F/I/I″ 승계).
    반환 (조건부 AUROC, 남은 S, 쌍, 빈수, **짝 단위 잔여 RR 격차**)."""
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0; ks_ = nb_ = 0; gsum = 0.0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_); nb_ += 1
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
        if pre_v is not None:                       # ★ 짝 단위 격차 (집합 중앙 아님)
            ps, pn = pre_v[m & tt], pre_v[m & ~tt]
            gsum += float(np.abs(ps[:, None] - pn[None, :]).sum())
    if den < 1:
        return float("nan"), ks_, 0.0, nb_, float("nan")
    gap = (gsum / den) / max(float(np.median(pre_v)), 1e-9) if pre_v is not None else float("nan")
    return num / den, ks_, den, nb_, gap

def exact_key(pre_v):
    return np.round(pre_v.astype(np.float64), 6)

run.log("\n" + "=" * 100)
run.log("【K-B】 정확 정합 — pre_rr 값이 **같은** 비트끼리만")
run.log("=" * 100)
EX = {a: np.full(len(RS), np.nan) for a in ARMS}
EOK = np.zeros(len(RS), bool); EGAP = np.full(len(RS), np.nan)
ESF = np.full(len(RS), np.nan); EBIN = np.full(len(RS), np.nan)
for i, r in enumerate(RS):
    tt = META[r]["tt"]; pre_m = META[r]["pre"]; key = exact_key(pre_m)
    for a in ARMS:
        sc_ = SCORES[r][a]
        if not np.isfinite(sc_).all():
            continue
        v, ks_, pr_, nb_, gp = grouped_auc(sc_, tt, key, MIN_BAND_S, MIN_BAND_N,
                                           pre_v=pre_m if a == "f1" else None)
        EX[a][i] = v
        if a == "f1":
            EOK[i] = bool(ks_ >= MIN_MATCH_S and pr_ >= MIN_MATCH_PAIR)
            EGAP[i] = gp; ESF[i] = ks_ / max(META[r]["pos"], 1); EBIN[i] = nb_
NEX = int(EOK.sum())
run.log(f"  정확 정합 가능 **{NEX}/{len(RS)}** 개체 · 남은 S 비율 중앙 "
        f"{np.nanmedian(ESF[EOK]) if EOK.any() else float('nan'):.3f} · "
        f"빈수 중앙 {np.nanmedian(EBIN[EOK]) if EOK.any() else float('nan'):.0f} · "
        f"**짝 단위 잔여 격차 {np.nanmedian(EGAP[EOK]) if EOK.any() else float('nan'):.6f}**")
if NEX < MIN_MATCH_REC:
    run.log(f"  ⛔ **정확 정합이 성립하지 않는다**({NEX} < {MIN_MATCH_REC}) — "
            "K1 이하를 읽지 않는다. 대역 정합으로 되돌리지 않는다(R16)")
EXACT_OK = NEX >= MIN_MATCH_REC
run.log("\n  정확 정합 조건 매크로 —")
for a in ARMS:
    run.log(f"    {a:<9} {np.nanmean(EX[a][EOK]) if EOK.any() else float('nan'):.4f}")
CONFIG["exact"] = dict(n_ok=NEX, pair_gap=float(np.nanmedian(EGAP[EOK])) if EOK.any() else None,
                       macro={a: float(np.nanmean(EX[a][EOK])) if EOK.any() else None for a in ARMS})
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【K-C】 ★ 정합 조건 **라벨셔플 null** — 각 팔의 자기 바닥
# f1·f3 같은 순수 특징은 셔플하면 0.5 다. 의미가 있는 건 **라벨을 쓰는 팔**
# (lr_* · stt) 이고, LR(전부) 0.778 이 교차적합 바닥인지 여기서 갈린다.
run.log("\n" + "=" * 100)
run.log(f"【K-C】 정합 조건 라벨셔플 null (셔플 {N_SHUF}회 · 정확 정합)")
run.log("=" * 100)
NULL = {a: np.full(len(RS), np.nan) for a in ARMS}
if EXACT_OK:
    for i, r in enumerate(RS):
        if not EOK[i]:
            continue
        tt = META[r]["tt"]; pre_m = META[r]["pre"]; key = exact_key(pre_m)
        X = META[r]["X"]; Bm = META[r]["B"]
        acc = {a: [] for a in ARMS}
        for s_ in range(N_SHUF):
            rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
            ts = rng.permutation(tt)                      # 유병률 보존
            Ss = build_scores(X, NAMES_G, ts, Bm, SEED0 + 31 * (s_ + 1))
            if Ss is None:
                continue
            for a in ARMS:
                sc_ = Ss[a]
                if not np.isfinite(sc_).all():
                    continue
                v, _, pr_, _, _ = grouped_auc(sc_, ts, key, MIN_BAND_S, MIN_BAND_N)
                if pr_ >= 1:
                    acc[a].append(v)
        for a in ARMS:
            if acc[a]:
                NULL[a][i] = float(np.nanmean(acc[a]))
    run.log("  팔별 — 실측 vs **셔플 null** vs 초과분")
    for a in ARMS:
        m_ = np.nanmean(EX[a][EOK]); n_ = np.nanmean(NULL[a][EOK])
        run.log(f"    {a:<9} {m_:.4f}  null {n_:.4f}  **초과 {m_-n_:+.4f}**")
    CONFIG["null"] = {a: float(np.nanmean(NULL[a][EOK])) for a in ARMS}
else:
    run.log("  ⛔ 정확 정합 불가 — 건너뛴다")
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【K-D】 관문 K1·K3·K4·K5 — **max 바닥을 폐기하고 초과분으로 비교**
def boot_diff(a, b, seed, nb=NB_BOOT, mask=None, const=None, q=2.5):
    a = np.asarray(a, float)
    d = (a - const) if const is not None else (a - np.asarray(b, float))
    if mask is not None:
        d = d[mask]
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log("【K-D】 관문 (정확 정합 · 개체 " + str(NEX) + ")")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

if not EXACT_OK:
    for k in ("K1", "K3", "K4", "K5", "K7"):
        g_(k, "⛔ 측정 불가", "정확 정합 성립 안 함")
else:
    # ── K1 ★ 정합기 감사 ② — 정확 정합이면 f1 은 동점뿐이다
    m1, lo1, hi1, n1_ = boot_diff(EX["f1"], None, SEED0 + 1, mask=EOK, const=0.0)
    DIFF["K1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
    g_("K1", decide(lo1, hi1, EXACT_TIE, "<"),
       f"정확 정합 f1ₘ **{m1:.4f}** [{lo1:.4f}, {hi1:.4f}] vs 상한 {EXACT_TIE}"
       f"   ← 동점뿐이라 0.5 여야 한다. 아니면 **정합기 결함**")
    if not VERD["K1"].startswith("✅"):
        run.log("    ⛔ **정확 정합인데도 f1 이 0.5 가 아니다 — 정합기·특징 정의를 먼저 고친다.**")
        run.log("       아래 관문을 「정합 조건」으로 읽지 않는다")

    # ── K3 ★ 절대 RR 을 맞춰도 **상대** 조기성은 안 맞는다
    m3, lo3, hi3, n3_ = boot_diff(EX["f2_16"], EX["f1"], SEED0 + 2, mask=EOK)
    DIFF["K3"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3_)
    g_("K3", decide(lo3, hi3, 0.0, ">"),
       f"짝지은 f2_16ₘ − f1ₘ **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}] · n={n3_}"
       f"   ← 지지면 **정합은 조기성을 통제한 적이 없다**(절대만 맞췄다)")

    # ── K4 J1 재판정 — max 안 쓴다. **자기 null 위 초과분**끼리, Bonferroni 2
    exc_f3 = EX["f3"] - 0.5
    exc_st = EX["stt"] - NULL["stt"]
    m4, lo4, hi4, n4_ = boot_diff(exc_f3, exc_st, SEED0 + 3, mask=EOK, q=BONF2 * 100)
    DIFF["K4"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
    g_("K4", decide(lo4, hi4, 0.0, ">"),
       f"(f3ₘ−0.5) − (STTₘ−nullSTTₘ) **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}] "
       f"· Bonferroni 2 · n={n4_}   ← **J1 재판정**(max 바닥 폐기)")
    m4b, lo4b, hi4b, _ = boot_diff(EX["f3"], EX["f1"], SEED0 + 4, mask=EOK, q=BONF2 * 100)
    run.log(f"    (대조 2/2) f3ₘ − f1ₘ {m4b:+.4f} [{lo4b:+.4f}, {hi4b:+.4f}]")

    # ── K5 LR(전부) 0.778 은 신호인가 교차적합 바닥인가
    m5, lo5, hi5, n5_ = boot_diff(EX["lr_all"], NULL["lr_all"], SEED0 + 5, mask=EOK)
    DIFF["K5"] = dict(mean=m5, lo=lo5, hi=hi5, n=n5_)
    g_("K5", decide(lo5, hi5, 0.0, ">"),
       f"LR(전부)ₘ − nullLR(전부)ₘ **{m5:+.4f}** [{lo5:+.4f}, {hi5:+.4f}] · n={n5_}"
       f"   ← Q7-I″ 에서 대역에 안 움직이던 0.778 의 정체")

    # ── K7 f5 의 짝은 f1 이 아니라 f3 다
    m7, lo7, hi7, n7_ = boot_diff(EX["lr_norr5"], EX["lr_norr"], SEED0 + 6, mask=EOK)
    DIFF["K7"] = dict(mean=m7, lo=lo7, hi=hi7, n=n7_)
    g_("K7", decide(lo7, hi7, 0.0, ">"),
       f"짝지은 LR(f3,f4,f5)ₘ − LR(f3,f4)ₘ **{m7:+.4f}** [{lo7:+.4f}, {hi7:+.4f}] · n={n7_}"
       f"   ← 3박자 패턴(정상–조기–휴지)의 나머지 절반")
    m7b, lo7b, hi7b, _ = boot_diff(EX["lr_i35"], EX["lr_norr5"], SEED0 + 7, mask=EOK)
    run.log(f"    (참고) f3×f5 상호작용 추가분 {m7b:+.4f} [{lo7b:+.4f}, {hi7b:+.4f}]")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【K-E】 정합기 감사 ③ — **고정 코호트** 대역 곡선
# Q7-I″ 곡선은 대역마다 개체가 달랐다(46→42→38→33). 정합이 촘촘해진 효과와
# 코호트가 바뀐 효과가 섞여 용량-반응이 아니었다. 여기선 **정확 정합 가능 개체로 고정**한다.
run.log("\n" + "=" * 100)
run.log("【K-E】 고정 코호트 대역 곡선 (같은 개체 · 짝 단위 잔여 격차)")
run.log("=" * 100)
FIX = EOK if EXACT_OK else np.zeros(len(RS), bool)
CURVE = {}
if FIX.any():
    for bf in list(BANDS) + [0.0]:
        M = {a: np.full(len(RS), np.nan) for a in ARMS}
        gap = np.full(len(RS), np.nan)
        for i, r in enumerate(RS):
            if not FIX[i]:
                continue
            tt = META[r]["tt"]; pre_m = META[r]["pre"]
            if bf == 0.0:
                key = exact_key(pre_m)
            else:
                bw = max(BAND_MIN, bf * float(np.median(pre_m)))
                key = np.floor(pre_m / bw).astype(np.int64)
            for a in ARMS:
                sc_ = SCORES[r][a]
                if not np.isfinite(sc_).all():
                    continue
                v, _, _, _, gp = grouped_auc(sc_, tt, key, MIN_BAND_S, MIN_BAND_N,
                                             pre_v=pre_m if a == "f1" else None)
                M[a][i] = v
                if a == "f1":
                    gap[i] = gp
        CURVE[bf] = dict(M=M, gap=gap)
        lab = "정확" if bf == 0.0 else f"{bf:.3f}"
        run.log(f"  대역 {lab:>5} — **짝 잔여 {np.nanmedian(gap[FIX]):.5f}** | "
                + " · ".join(f"{a} {np.nanmean(M[a][FIX]):.4f}"
                             for a in ("f1", "f2_16", "f3", "stt", "lr_norr", "lr_all")))
    run.log(f"\n  ▸ 개체를 **{int(FIX.sum())}개로 고정**했으므로 이 곡선은 용량-반응이다.")
    run.log("    Q7-I″ 곡선(46→33)은 코호트 변화가 섞여 있었다 — 그 곡선의 단조성은 근거가 아니다")
    CONFIG["curve_fixed"] = {("exact" if bf == 0.0 else f"{bf:.3f}"): dict(
        gap=float(np.nanmedian(CURVE[bf]["gap"][FIX])),
        macro={a: float(np.nanmean(CURVE[bf]["M"][a][FIX])) for a in ARMS})
        for bf in CURVE}
else:
    run.log("  ⛔ 고정 코호트 없음 — 건너뛴다")
run.save_json("config", CONFIG)

In [ ]:
# CELL 9 — 【K-F】 K6 — J4 를 **혼합 / 런 따로** 재판정 + f6 구제 시험
# Q7-I″ J4 는 혼합(12)+런(5) 을 한 층으로 묶어 −0.0718 을 냈다. 두 층의 방향이
# 반대면 그 평균은 아무 것도 뜻하지 않는다 → **분리해서 다시 묻는다.**
run.log("\n" + "=" * 100)
run.log("【K-F】 K6 — 층 분리 재판정 (혼합 / 런) · f6 이 런을 구하나")
run.log("=" * 100)
ISOF = np.array([META[r]["iso_frac"] for r in RS])
MXR  = np.array([META[r]["max_run"] for r in RS])
L_ISO = ISOF >= ISO_HI
L_RUN = ISOF <= ISO_LO
L_MIX = ~L_ISO & ~L_RUN
run.log(f"  층 크기 — 고립 {int(L_ISO.sum())} · 혼합 {int(L_MIX.sum())} · "
        f"런 {int(L_RUN.sum())} (최장런 중앙 {np.median(MXR[L_RUN]) if L_RUN.any() else float('nan'):.0f})")
for nm_, msk in (("고립 S", L_ISO), ("혼합", L_MIX), ("런 우세", L_RUN)):
    if int(msk.sum()) < 3:
        run.log(f"  {nm_:<8} {int(msk.sum())}개체 — 3 미만이라 CI 를 내지 않는다")
        continue
    a_, l_, h_, n_ = boot_diff(RAW["f2_16"], RAW["f1"], SEED0 + 11, mask=msk)
    b_, l2, h2, _ = boot_diff(RAW["lr_f1f6"], RAW["lr_f1"], SEED0 + 12, mask=msk)
    run.log(f"  {nm_:<8} {n_:>2}개체 · f1 {np.nanmean(RAW['f1'][msk]):.4f} · "
            f"f2_16 {np.nanmean(RAW['f2_16'][msk]):.4f}")
    run.log(f"           f2_16 − f1 **{a_:+.4f}** [{l_:+.4f}, {h_:+.4f}]  |  "
            f"LR(f1,f6) − LR(f1) **{b_:+.4f}** [{l2:+.4f}, {h2:+.4f}]")
    VERD[f"K6_{nm_}"] = decide(l_, h_, 0.0, ">")
    DIFF[f"K6_{nm_}"] = dict(mean=a_, lo=l_, hi=h_, n=n_,
                             f6_mean=b_, f6_lo=l2, f6_hi=h2)
run.log("\n  ▸ 혼합과 런의 **부호가 반대**면 Q7-I″ 의 J4(−0.0718)는 두 층의 평균일 뿐이고,")
run.log("    `f2` 를 폐기할 근거가 아니다. 런에서 죽는 건 국소창(k=8·16)이 런 안에 갇히기")
run.log("    때문이며, 그 가설의 라벨프리 처방이 **f6**(국소 기저선의 전역 대비 이동)이다")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 10 — 【K-G】 그림 · 관문 요약
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다(네모 방지).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
if CURVE:
    xs = [bf if bf > 0 else 0.004 for bf in CURVE]
    for a_, c_, lb in (("f1", "tab:red", "f1 (must be 0.5)"), ("f2_16", "tab:purple", "f2_16"),
                       ("f3", "tab:green", "f3"), ("stt", "tab:orange", "STT"),
                       ("lr_all", "tab:blue", "LR(all)")):
        ax[0].plot(xs, [np.nanmean(CURVE[bf]["M"][a_][FIX]) for bf in CURVE],
                   "o-", color=c_, label=lb)
    ax[0].axhline(0.5, ls="--", lw=0.8, color="crimson")
    ax[0].set_xscale("log"); ax[0].invert_xaxis()
    ax[0].set_xlabel("band width  (0.004 = exact match)  <- narrower")
    ax[0].set_ylabel("matched macro AUROC (fixed cohort)")
    ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    ax[1].plot(xs, [np.nanmedian(CURVE[bf]["gap"][FIX]) for bf in CURVE], "s-", color="crimson")
    ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].invert_xaxis()
    ax[1].set_xlabel("band width  <- narrower")
    ax[1].set_ylabel("PAIRWISE residual RR gap")
    ax[1].grid(alpha=.3)
if EXACT_OK:
    aa = [a for a in ARMS if a not in ("f2_8",)]
    ax[2].barh(range(len(aa)), [np.nanmean(EX[a][EOK]) - np.nanmean(NULL[a][EOK]) for a in aa],
               color="tab:blue")
    ax[2].set_yticks(range(len(aa))); ax[2].set_yticklabels(aa, fontsize=7)
    ax[2].axvline(0, color="k", lw=.8)
    ax[2].set_xlabel("excess over own label-shuffle null (exact match)")
    ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
run.save_fig("q7i3_match_exact", fig)

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in sorted(VERD):
    run.log(f"  {k:<12}{VERD[k]}")
if EXACT_OK and not VERD.get("K1", "").startswith("✅"):
    run.log("\n  ⛔ **정확 정합에서도 f1 이 0.5 가 아니다** — 정합 해석 전체가 여기서 멈춘다.")
    run.log("     이건 데이터 성질이 아니라 **특징·정합기 정의 문제**일 수 있고, 그걸 가른 뒤에야")
    run.log("     f3·STT·LR 값을 「정합 조건」이라고 부를 수 있다")
elif EXACT_OK:
    run.log("\n  ✅ 정확 정합에서 f1 이 무너졌다 — 이 조건의 값들은 **조기성 통제 후**로 읽어도 된다")
CONFIG["gates"] = VERD
run.save_json("config", CONFIG)
run.finish()
run.log("완료")